In [2]:
# Installs the homework's packages. Safe to re-run.
import sys, subprocess
pkgs = ["numpy", "pandas", "matplotlib", "scikit-learn", "torch"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("Done. If you saw errors about 'torch', see https://pytorch.org/get-started/locally/")

Done. If you saw errors about 'torch', see https://pytorch.org/get-started/locally/


In [3]:
# Installs the homework's packages. Safe to re-run.
import sys, subprocess
pkgs = ["numpy", "pandas", "matplotlib", "scikit-learn", "torch"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("Done. If you saw errors about 'torch', see https://pytorch.org/get-started/locally/")

Done. If you saw errors about 'torch', see https://pytorch.org/get-started/locally/


In [4]:
# Quick environment check
import importlib
for name in ["numpy", "pandas", "matplotlib", "sklearn", "torch"]:
    try:
        m = importlib.import_module(name)
        print(f"OK  {name:12s} {getattr(m, '__version__', '')}")
    except ImportError:
        print(f"MISSING  {name}  <- run the install cell above")

OK  numpy        2.4.6
OK  pandas       3.0.3
OK  matplotlib   3.11.0
OK  sklearn      1.9.0
OK  torch        2.12.1+cpu


In [5]:
import numpy as np
import torch


torch.manual_seed(0)
np.random.seed(0)

print("torch", torch.__version__)
print("Setup OK")

torch 2.12.1+cpu
Setup OK


In [6]:
# Data + plotting helpers (defined in hw1_tests.py):
import pandas as pd

df = pd.read_csv("station07.csv")

df.head()

,date_time,nwp_globalirrad,nwp_directirrad,nwp_temperature,nwp_humidity,nwp_windspeed,nwp_winddirection,nwp_pressure,lmd_totalirrad,lmd_diffuseirrad,lmd_temperature,lmd_pressure,lmd_winddirection,lmd_windspeed,power
0,2018-06-30 16:00:00,0.0,0.0,19.86,61.42,3.49,338.26,930.57,0,0,17.600000,934.799988,168,0.0,0.0
1,2018-06-30 16:15:00,0.0,0.0,19.81,61.22,3.51,339.19,930.51,0,0,17.500000,934.700012,172,0.0,0.0
2,2018-06-30 16:30:00,0.0,0.0,19.76,61.03,3.52,340.19,930.55,0,0,17.299999,934.599976,174,0.0,0.0
3,2018-06-30 16:45:00,0.0,0.0,19.70,60.91,3.53,340.14,930.47,0,0,17.200001,934.599976,173,0.0,0.0
4,2018-06-30 17:00:00,0.0,0.0,19.60,60.89,3.51,339.90,930.19,0,0,17.100000,934.599976,175,0.0,0.0


In [7]:
df = pd.read_csv("station07.csv")

df["date_time"] = pd.to_datetime(df["date_time"])

df = df.set_index("date_time")

In [8]:
station07_theoretical = pd.read_csv(
    "station07_theoretical.csv",
    index_col="date_time",
    parse_dates=True
)

In [9]:
df.index = pd.DatetimeIndex(df.index).tz_localize("UTC")

station07_theoretical.index = pd.DatetimeIndex(
    station07_theoretical.index
).tz_convert("UTC")

In [10]:
df = df.join(
    station07_theoretical,
    how="left"
)

In [11]:
df[[
    "power",
    "P_theoretical_W"
]].head(20)

,power,P_theoretical_W
date_time,,
2018-06-30 16:00:00+00:00,0.0,0.0
2018-06-30 16:15:00+00:00,0.0,0.0
2018-06-30 16:30:00+00:00,0.0,0.0
2018-06-30 16:45:00+00:00,0.0,0.0
2018-06-30 17:00:00+00:00,0.0,0.0
2018-06-30 17:15:00+00:00,0.0,0.0
2018-06-30 17:30:00+00:00,0.0,0.0
2018-06-30 17:45:00+00:00,0.0,0.0
2018-06-30 18:00:00+00:00,0.0,0.0


In [12]:
df["power_W"] = df["power"] * 1_000_000

In [13]:
df["K_PV"] = np.where(
    (df["POA_clear"] >= 200) &
    (df["P_theoretical_W"] > 0),
    df["power_W"] / df["P_theoretical_W"],
    np.nan
)

In [14]:
df[[
    "power_W",
    "P_theoretical_W",
    "K_PV"
]].describe()

,power_W,P_theoretical_W,K_PV
count,3.292800e+04,3.292800e+04,13182.000000
mean,2.618584e+06,5.748727e+06,0.443783
std,4.106109e+06,7.436627e+06,0.232746
min,0.000000e+00,0.000000e+00,0.000000
25%,0.000000e+00,0.000000e+00,0.241033
50%,0.000000e+00,1.363454e+02,0.487356
75%,4.206456e+06,1.290845e+07,0.633734
max,1.728058e+07,2.000000e+07,1.036025


In [15]:
df.loc[
    df["K_PV"].idxmax(),
    [
        "power_W",
        "P_theoretical_W",
        "K_PV",
        "GHI_clear",
        "DNI_clear",
        "DHI_clear",
        "POA_clear"
    ]
]

power_W            5.087538e+06
P_theoretical_W    4.910631e+06
K_PV               1.036025e+00
GHI_clear          3.410910e+02
DNI_clear          6.027238e+02
DHI_clear          8.603880e+01
POA_clear          2.505424e+02
Name: 2018-07-04 09:30:00+00:00, dtype: float64

In [16]:
df.to_csv("station07_with_K.csv", index=True)

In [17]:
df1 = pd.read_csv("station07_with_K.csv")

In [18]:
df1.insert(
    len(df1.columns) - 1,
    "GHI_ratio",
    df1["nwp_globalirrad"] / df1["GHI_clear"]
)

In [19]:
df1.insert(
    len(df1.columns) - 1,
    "DNI_ratio",
    np.where(
        df1["DNI_clear"] > 50,
        df1["nwp_directirrad"] / df1["DNI_clear"],
        np.nan
    )
)

In [20]:
df_clean = df1[df1["P_theoretical_W"] > 500000].copy()

In [21]:
df_clean.loc[
    df_clean["K_PV"].idxmax(),
    [
        "power_W",
        "P_theoretical_W",
        "K_PV",
        "GHI_clear",
        "DNI_clear",
        "DHI_clear",
        "POA_clear"
    ]
]

power_W            5.087538e+06
P_theoretical_W    4.910631e+06
K_PV               1.036025e+00
GHI_clear          3.410910e+02
DNI_clear          6.027238e+02
DHI_clear          8.603880e+01
POA_clear          2.505424e+02
Name: 358, dtype: float64

In [22]:
df_clean.to_csv("station07_with_K_ratios.csv", index=True)